### **01 Ingest Raw CSV to Parquet (Bronze Layer)**


In [1]:
import os
import time
from pathlib import Path
import pandas as pd
import pyarrow as pa
import pyarrow.csv as pv
import pyarrow.parquet as pq
import gc
import psutil
import shutil

print("Libraries successfully imported!")
print(f"PyArrow version: {pa.__version__}")
print(f"Pandas version: {pd.__version__}")

Libraries successfully imported!
PyArrow version: 21.0.0
Pandas version: 2.3.2


In [2]:
# Dynamically resolve project root
current_path = Path.cwd().resolve()
project_root = None
for p in [current_path] + list(current_path.parents):
    if (p / "inventory-lakehouse").is_dir():
        project_root = p
        break

if project_root is None:
    project_root = Path("D:/Demand_forecasting").resolve()

SOURCES_DIR = project_root / "sources"
BRONZE_DIR = project_root / "inventory-lakehouse" / "data_lake" / "bronze"

# Ensure Bronze directory exists
BRONZE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project Root: {project_root}")
print(f"Sources Dir:  {SOURCES_DIR}")
print(f"Bronze Dir:   {BRONZE_DIR}")


Project Root: D:\Demand_forecasting
Sources Dir:  D:\Demand_forecasting\sources
Bronze Dir:   D:\Demand_forecasting\inventory-lakehouse\data_lake\bronze


In [3]:
def format_size(bytes_size):
    """Helper to format file size in human-readable units."""
    for unit in ['B', 'KB', 'MB', 'GB', 'TB']:
        if bytes_size < 1024:
            return f"{bytes_size:.2f} {unit}"
        bytes_size /= 1024
    return f"{bytes_size:.2f} TB"

In [4]:
def ingest_csv_to_parquet(csv_path: Path, parquet_path: Path) -> dict:
    """
    Ingests a CSV file into a Parquet file using PyArrow streaming.
    Tracks row counts and logs schema details.
    For extremely large datasets (like train.csv), writes in 20M-row partitioned segments.
    """
    print(f"\n{'='*60}")
    print(f"Processing: {csv_path.name}")
    print(f"{'='*60}")
    start_time = time.time()
    process = psutil.Process(os.getpid())
    
    # Explicitly specify column types to prevent inference errors on large datasets
    column_types = {
        "id": pa.int64(),
        "store_nbr": pa.int32(),
        "item_nbr": pa.int32(),
        "unit_sales": pa.float64(),
        "onpromotion": pa.bool_(),
        "transactions": pa.int32(),
        "perishable": pa.int32(),
        "class": pa.int32(),
        "cluster": pa.int32(),
        "dcoilwtico": pa.float64(),
    }
    
    # Set options for high throughput and memory safety
    # 4 MB block size for streaming reader to limit RAM overhead
    read_options = pv.ReadOptions(block_size=4 * 1024 * 1024)
    convert_options = pv.ConvertOptions(
        null_values=["", "NA", "NaN", "null", "None"],
        column_types=column_types
    )
    
    csv_row_count = 0
    batch_count = 0
    
    # Set up segmented writes for train.csv
    is_segmented = (csv_path.name == "train.csv")
    if is_segmented:
        parquet_path.mkdir(parents=True, exist_ok=True)
        def get_part_path(idx):
            return parquet_path / f"part_{idx}.parquet"
        part_idx = 0
        current_parquet_path = get_part_path(part_idx)
        part_row_count = 0
    else:
        current_parquet_path = parquet_path
        
    writer = None
    
    try:
        # Open the CSV as a streaming reader
        with pv.open_csv(csv_path, read_options=read_options, convert_options=convert_options) as reader:
            schema = reader.schema
            print(f"Inferred Schema:")
            for field in schema:
                print(f"  - {field.name}: {field.type}")
                
            # Open initial Parquet writer
            writer = pq.ParquetWriter(current_parquet_path, schema, compression='snappy')
            
            for batch in reader:
                writer.write_batch(batch)
                csv_row_count += batch.num_rows
                batch_count += 1
                
                # Clean up memory references immediately
                del batch
                
                # Clean memory every 10 batches
                if batch_count % 10 == 0:
                    gc.collect()
                    pa.default_memory_pool().release_unused()
                    
                # Rollover for segmented files (train.csv) every 20 million rows
                if is_segmented:
                    part_row_count = csv_row_count - (part_idx * 20_000_000)
                    if part_row_count >= 20_000_000:
                        writer.close()
                        print(f"  - Closed part_{part_idx}.parquet with {part_row_count:,} rows.")
                        
                        part_idx += 1
                        current_parquet_path = get_part_path(part_idx)
                        writer = pq.ParquetWriter(current_parquet_path, schema, compression='snappy')
                        
                        gc.collect()
                        pa.default_memory_pool().release_unused()
                        
                # Log memory usage and progress
                if batch_count % 100 == 0:
                    mem_usage = process.memory_info().rss
                    print(f"  - Written {csv_row_count:,} rows. RAM Usage: {format_size(mem_usage)}")
                    
        # Close the final writer
        if writer is not None:
            writer.close()
            if is_segmented:
                final_part_rows = csv_row_count - (part_idx * 20_000_000)
                print(f"  - Closed final part_{part_idx}.parquet with {final_part_rows:,} rows.")
                
        # Final clean up for this file
        gc.collect()
        pa.default_memory_pool().release_unused()
        
        elapsed_time = time.time() - start_time
        
        # Row validation across segments or single file
        if is_segmented:
            parquet_row_count = 0
            parquet_files = sorted(list(parquet_path.glob("*.parquet")))
            for pf in parquet_files:
                meta = pq.read_metadata(pf)
                parquet_row_count += meta.num_rows
            parquet_size = sum(f.stat().st_size for f in parquet_files)
        else:
            parquet_metadata = pq.read_metadata(parquet_path)
            parquet_row_count = parquet_metadata.num_rows
            parquet_size = parquet_path.stat().st_size
            
        # Raise error if counts do not match
        if csv_row_count != parquet_row_count:
            raise ValueError(
                f"Row count mismatch for {csv_path.name}! "
                f"CSV stream counted: {csv_row_count:,}, Parquet: {parquet_row_count:,}"
            )
            
        csv_size = csv_path.stat().st_size
        compression_ratio = csv_size / parquet_size if parquet_size > 0 else 0
        
        print(f"\nIngestion complete!")
        print(f"Duration:          {elapsed_time:.2f} seconds")
        print(f"Rows Ingested:     {csv_row_count:,}")
        print(f"CSV File Size:     {format_size(csv_size)}")
        print(f"Parquet File Size: {format_size(parquet_size)}")
        print(f"Compression Ratio: {compression_ratio:.2f}x")
        
        return {
            "Dataset": csv_path.stem,
            "Inferred Columns": len(schema.names),
            "Rows Counted": csv_row_count,
            "CSV Size": format_size(csv_size),
            "Parquet Size": format_size(parquet_size),
            "Compression Ratio": f"{compression_ratio:.2f}x",
            "Duration (s)": f"{elapsed_time:.2f}s",
            "Status": "SUCCESS"
        }
    except Exception as e:
        if writer is not None:
            try:
                writer.close()
            except Exception: 
                pass
        raise e

In [5]:
# Find all raw CSV files in sources directory (ignoring .7z files)
csv_files = sorted(list(SOURCES_DIR.glob("*.csv")))

print(f"Found {len(csv_files)} CSV files to ingest:")
for f in csv_files:
    print(f"  - {f.name} ({format_size(f.stat().st_size)})")

Found 8 CSV files to ingest:
  - holidays_events.csv (21.79 KB)
  - items.csv (99.45 KB)
  - oil.csv (20.10 KB)
  - sample_submission.csv (38.57 MB)
  - stores.csv (1.35 KB)
  - test.csv (120.32 MB)
  - train.csv (4.65 GB)
  - transactions.csv (1.48 MB)


In [6]:
results = []
for csv_file in csv_files:
    parquet_file = BRONZE_DIR / f"{csv_file.stem}.parquet"
    try:
        stats = ingest_csv_to_parquet(csv_file, parquet_file)
        results.append(stats)
    except Exception as e:
        print(f"Error processing {csv_file.name}: {str(e)}")
        results.append({
            "Dataset": csv_file.stem,
            "Inferred Columns": 0,
            "Rows Counted": 0,
            "CSV Size": format_size(csv_file.stat().st_size),
            "Parquet Size": "N/A",
            "Compression Ratio": "N/A",
            "Duration (s)": "0.00s",
            "Status": f"FAILED: {str(e)}"
        })


Processing: holidays_events.csv
Inferred Schema:
  - date: date32[day]
  - type: string
  - locale: string
  - locale_name: string
  - description: string
  - transferred: bool

Ingestion complete!
Duration:          0.02 seconds
Rows Ingested:     350
CSV File Size:     21.79 KB
Parquet File Size: 5.42 KB
Compression Ratio: 4.02x

Processing: items.csv
Inferred Schema:
  - item_nbr: int32
  - family: string
  - class: int32
  - perishable: int32

Ingestion complete!
Duration:          0.01 seconds
Rows Ingested:     4,100
CSV File Size:     99.45 KB
Parquet File Size: 32.55 KB
Compression Ratio: 3.06x

Processing: oil.csv
Inferred Schema:
  - date: date32[day]
  - dcoilwtico: double

Ingestion complete!
Duration:          0.02 seconds
Rows Ingested:     1,218
CSV File Size:     20.10 KB
Parquet File Size: 13.10 KB
Compression Ratio: 1.53x

Processing: sample_submission.csv
Inferred Schema:
  - id: int64
  - unit_sales: double

Ingestion complete!
Duration:          0.44 seconds
Rows 

In [7]:
# Display summary table
df_summary = pd.DataFrame(results)
print("\n" + "#" * 60)
print("                 BRONZE INGESTION SUMMARY REPORT")
print("#" * 60)
print(df_summary.to_string(index=False))


############################################################
                 BRONZE INGESTION SUMMARY REPORT
############################################################
          Dataset  Inferred Columns  Rows Counted  CSV Size Parquet Size Compression Ratio Duration (s)  Status
  holidays_events                 6           350  21.79 KB      5.42 KB             4.02x        0.02s SUCCESS
            items                 4          4100  99.45 KB     32.55 KB             3.06x        0.01s SUCCESS
              oil                 2          1218  20.10 KB     13.10 KB             1.53x        0.02s SUCCESS
sample_submission                 2       3370464  38.57 MB     15.53 MB             2.48x        0.44s SUCCESS
           stores                 5            54   1.35 KB      2.26 KB             0.60x        0.02s SUCCESS
             test                 5       3370464 120.32 MB     24.72 MB             4.87x        0.90s SUCCESS
            train                 6     1254